# PAYBACK Lightweight Assistant — Demo Notebook

This notebook walks through the assistant's capabilities with seven carefully chosen
queries that together exercise every component of the system.

## What this system is

A multilingual (German/English) shopping assistant for the PAYBACK app. Takes a
natural-language query, classifies intent, optionally expands basket-style queries,
retrieves products from three partner catalogs (dm, EDEKA, Amazon), applies
loyalty-aware ranking, and returns either recommendations, a clarifying question,
or a navigation target.

## Architecture (top to bottom)

```
User query (EN/DE)
       ↓
Intent Agent (Claude tool-use)
       ↓
       ├── navigational + partner       → return navigation target
       ├── support                      → return out-of-scope clarification
       ├── navigational without partner → return partner picker
       ├── specific + high-confidence:
       │     ├── basket query? → Query Expander (catalog-grounded)
       │     │                    → multi-query retrieval with relevance filter
       │     └── single query? → single retrieval
       │     → Loyalty Ranker → return recommendations
       └── vague or low-confidence     → catalog-grounded clarification
```

## What you'll see in this demo

| # | Query | What it demonstrates |
|---|---|---|
| 1 | `wireless mouse` | Basic specific search, single-item retrieval |
| 2 | `something for my dog` | Vague query → catalog-grounded clarification |
| 3 | `essentials for moving into a new house` | Basket query → expansion + dropped-query transparency |
| 4 | Same as 3, with `user_edeka_heavy` | Loyalty signal — diversity bonus surfaces under-used partners |
| 5 | `Windeln bei dm` | German cross-lingual + partner-mention extraction |
| 6 | `take me to the shop` | Navigational without partner → fast hardcoded clarification |
| 7 | `how do I redeem my points` | Support intent → graceful out-of-scope |

In [1]:
import sys, os, pathlib
# Ensure repo root is on sys.path regardless of CWD
_repo = pathlib.Path(os.getcwd())
if not (_repo / 'app').exists():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))
os.chdir(_repo)

import asyncio
import json
from app.agents.router import get_default_router
from app.models.schemas import UserContext, Partner

router = get_default_router()

def pretty(response):
    """Print key fields then format recommendations/clarification/navigation."""
    intent = response.intent_result
    print(f"Response type:  {response.response_type}")
    print(f"Language:       {intent.language.value}")
    print(f"Intent:         {intent.intent.value}")
    print(f"Specificity:    {intent.specificity.value} (confidence {intent.confidence:.2f})")
    print(f"Extracted query:  {intent.extracted_query}")
    print(f"Target partner: {intent.target_partner}")
    print(f"Is basket:      {intent.is_basket_query}")
    print(f"Deals preferred: {intent.prefers_deals}")
    print(f"Reasoning:      {intent.reasoning}")
    print(f"Latency:        {response.latency_ms:.0f} ms")
    print(f"Cost:           €{response.estimated_cost_eur:.5f}")

    if response.response_type == "recommendations":
        print(f"\nTop {min(5, len(response.recommendations))} recommendations:")
        for rec in response.recommendations[:5]:
            print(f"  {rec.rank}. [{rec.product.partner.value:6}] "
                  f"{rec.product.name[:55]:55s} "
                  f"(sem={rec.semantic_score:.2f}, "
                  f"loy={rec.loyalty_boost:.2f}, "
                  f"div={rec.diversity_bonus:.2f}, "
                  f"final={rec.final_score:.2f})")
        if response.debug_expanded_queries:
            print(f"\nExpansion proposed: {response.debug_expanded_queries}")
        if response.debug_dropped_queries:
            print(f"Expansion dropped:  {response.debug_dropped_queries}")
    elif response.response_type == "clarification":
        print(f"\nClarifying question: {response.clarification.question}")
        print(f"Options: {response.clarification.suggested_options}")
    elif response.response_type == "navigation":
        print(f"\nNavigate to: {response.navigation_target}")

print("✓ Router initialized and ready")

✓ Router initialized and ready


## Query 1 — Basic specific search

Single-item English query. Expectation: retrieval returns Amazon electronics products.
Single-query path (no expansion). Cold-start ranking (no user_id).

In [2]:
response = await router.handle(query="wireless mouse")
pretty(response)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Response type:  recommendations
Language:       en
Intent:         search
Specificity:    specific (confidence 0.99)
Extracted query:  wireless mouse
Target partner: amazon
Is basket:      False
Deals preferred: False
Reasoning:      The query is a clear, concrete product search in English for a single item with no partner mentioned, no deal-seeking language, and no basket intent.
Latency:        16236 ms
Cost:           €0.00320

Top 5 recommendations:
  1. [amazon] Logitech MX Master 3S Wireless Mouse                    (sem=0.74, loy=0.70, div=0.00, final=0.65)
  2. [amazon] Logitech MX Master 3S Wireless Mouse                    (sem=0.72, loy=0.70, div=0.00, final=0.64)
  3. [amazon] Logitech MX Master 3S Wireless Mouse                    (sem=0.72, loy=0.70, div=0.00, final=0.64)
  4. [amazon] Logitech MX Master 3S Wireless Mouse                    (sem=0.71, loy=0.70, div=0.00, final=0.64)
  5. [amazon] Logitech MX Master 3S Wireless Mouse                    (sem=0.71, loy=0.70,

## Query 2 — Vague query → catalog-grounded clarification

The query is too vague to retrieve meaningful results. The system should NOT guess —
it should ask a clarifying question grounded in what the catalog actually returned.

In [3]:
response = await router.handle(query="something for my dog")
pretty(response)

Response type:  clarification
Language:       en
Intent:         search
Specificity:    vague (confidence 0.92)
Extracted query:  dog products
Target partner: None
Is basket:      False
Deals preferred: False
Reasoning:      The user wants a product for their dog but provides no concrete category, product type, or brand, making this too broad to retrieve useful results — classified as vague search with no partner or deal signals.
Latency:        7364 ms
Cost:           €0.00627

Clarifying question: What are you looking for for your dog?
Options: ['Food & Treats', 'Care & Grooming', 'Accessories & Toys', 'Health & Supplements']


## Query 3 — Basket query, cold start

A multi-item intent. The intent agent flags `is_basket_query=True`, triggering the
query expander. The expander does a catalog pre-flight to discover available categories,
then proposes 3-5 sub-queries within those categories.

After expansion, each sub-query retrieves products with a relevance threshold filter —
sub-queries returning nothing above threshold are dropped silently. The dropped
sub-queries are surfaced in `debug_dropped_queries` for transparency.

Watch for:
- `is_basket_query=True` in the intent result
- The expansion proposed vs. dropped lists
- Mixed-partner basket (dm cleaning + Amazon storage/tools)
- Cold-start diversity bonus computed from result-set distribution

In [14]:
response = await router.handle(query="essentials for moving into a new house")
pretty(response)

Response type:  recommendations
Language:       en
Intent:         search
Specificity:    specific (confidence 0.87)
Extracted query:  essentials for moving into a new house
Target partner: None
Is basket:      True
Deals preferred: False
Reasoning:      The query implies a clear use-case (moving into a new home) with a well-understood set of products typically bought together (cleaning supplies, storage, kitchenware, toiletries, etc.), making it specific enough for retrieval and a strong basket query, with no deal signals or partner mentioned.
Latency:        11301 ms
Cost:           €0.00516

Top 5 recommendations:
  1. [dm    ] Denkmit Spülmaschinentabs Classic                       (sem=0.62, loy=0.30, div=0.47, final=0.51)
  2. [dm    ] Denkmit Spülmaschinentabs Power Multi                   (sem=0.55, loy=0.40, div=0.47, final=0.50)
  3. [amazon] WD Elements Portable 2TB External HDD                   (sem=0.51, loy=0.40, div=0.53, final=0.48)
  4. [amazon] WD Elements Desktop 8T

## Query 4 — Same query, with an EDEKA-heavy user

The same query as Query 3, but now with `user_id="user_edeka_heavy"`. The user
shops EDEKA 70% of the time, dm 20%, Amazon 10%.

The loyalty ranker uses the affinity profile to compute a diversity bonus that
*inversely* weights partners — boosting dm and Amazon products to surface
partners this user under-uses. PAYBACK's business model rewards diversification.

Compare to Query 3 — same intent, same expansion, same retrieval. The ranker
re-orders based on user signal.

In [5]:
edeka_heavy = UserContext(
    user_id="user_edeka_heavy",
    partner_affinity={Partner.dm: 0.2, Partner.edeka: 0.7, Partner.amazon: 0.1},
    is_new_user=False,
)

response = await router.handle(
    query="essentials for moving into a new house",
    user_context=edeka_heavy,
)
pretty(response)

Response type:  recommendations
Language:       en
Intent:         search
Specificity:    specific (confidence 0.88)
Extracted query:  essentials for moving into a new house
Target partner: None
Is basket:      True
Deals preferred: False
Reasoning:      The query implies a concrete shopping scenario (moving into a new home) that maps to a well-understood basket of products (cleaning supplies, storage, kitchenware, toiletries, etc.), making it specific enough for retrieval and clearly a multi-product basket query; no partner is mentioned and no deal-seeking language is present.
Latency:        9525 ms
Cost:           €0.00516

Top 5 recommendations:
  1. [amazon] Amazon Basics 173-teiliges Werkzeugset                  (sem=0.51, loy=1.10, div=0.45, final=0.68)
  2. [amazon] Amazon Basics Werkzeugkoffer 65-teilig                  (sem=0.46, loy=0.70, div=0.45, final=0.53)
  3. [amazon] WD Elements Portable 2TB External HDD                   (sem=0.51, loy=0.40, div=0.45, final=0.47)
  4

## Query 5 — German query with partner mention

`Windeln bei dm` ("diapers at dm" in German). This exercises three things at once:

- **Language detection**: should be `de`
- **Cross-lingual retrieval**: the embedding model is multilingual end-to-end —
  German queries retrieve from German product descriptions without translation
- **Partner mention extraction**: `target_partner=dm`, which constrains retrieval
  to dm's catalog only

Expectation: all results from dm, all genuine diaper products.

In [15]:
response = await router.handle(query="Windeln bei dm")
pretty(response)

Response type:  recommendations
Language:       de
Intent:         search
Specificity:    specific (confidence 0.97)
Extracted query:  Windeln
Target partner: dm
Is basket:      False
Deals preferred: False
Reasoning:      The user is searching for a specific product (diapers/"Windeln") at a named, recognized partner (dm), making it a specific search query with dm as target_partner; no deal-seeking language and no basket implied.
Latency:        6738 ms
Cost:           €0.00320

Top 5 recommendations:
  1. [dm    ] Pampers Premium Protection Gr. 3                        (sem=0.27, loy=0.70, div=0.00, final=0.37)
  2. [dm    ] babylove Windeln Größe 4                                (sem=0.45, loy=0.30, div=0.00, final=0.36)
  3. [dm    ] babylove Premium Windeln Gr. 4                          (sem=0.55, loy=0.00, div=0.00, final=0.33)
  4. [dm    ] babylove Windeln Premium Gr. 3                          (sem=0.51, loy=0.00, div=0.00, final=0.31)
  5. [dm    ] Persil Universal Megaperls 

In [16]:
response = await router.handle(query="Reinigung bei Rewe")
pretty(response)

Response type:  recommendations
Language:       de
Intent:         search
Specificity:    specific (confidence 0.82)
Extracted query:  Reinigung
Target partner: None
Is basket:      False
Deals preferred: False
Reasoning:      The user searches for cleaning products ("Reinigung") and mentions "Rewe," which is not a supported partner in our system (not dm, edeka, or amazon), so target_partner is set to null and the search spans our available partners; "Reinigung" is specific enough to retrieve useful products.
Latency:        6289 ms
Cost:           €0.00320

Top 5 recommendations:
  1. [dm    ] Sagrotan Hygiene-Spray                                  (sem=0.55, loy=1.10, div=0.00, final=0.66)
  2. [dm    ] Persil Universal Waschmittel                            (sem=0.52, loy=0.40, div=0.00, final=0.43)
  3. [dm    ] Denkmit Spülmaschinentabs Classic                       (sem=0.57, loy=0.30, div=0.00, final=0.43)
  4. [dm    ] Persil Universal Megaperls                              (se

In [19]:
response = await router.handle(query="Vitamin D Tabletten")
pretty(response)

Response type:  recommendations
Language:       de
Intent:         search
Specificity:    specific (confidence 0.97)
Extracted query:  Vitamin D Tabletten
Target partner: None
Is basket:      False
Deals preferred: False
Reasoning:      The query is a clear, concrete German product search for a specific supplement type (Vitamin D tablets), with no partner mentioned, no deal-seeking language, and no basket or navigational signals.
Latency:        5065 ms
Cost:           €0.00320

Top 5 recommendations:
  1. [dm    ] tetesept Vitamin D3 + K2 Tropfen                        (sem=0.71, loy=0.40, div=0.00, final=0.55)
  2. [dm    ] DAS gesunde PLUS Vitamin D3                             (sem=0.81, loy=0.00, div=0.00, final=0.49)
  3. [dm    ] Das gesunde Plus Vitamin D3 Tabletten                   (sem=0.80, loy=0.00, div=0.00, final=0.48)
  4. [dm    ] tetesept Vitamin D3 1000 I.E.                           (sem=0.79, loy=0.00, div=0.00, final=0.47)
  5. [dm    ] tetesept Vitamin D3 2000 I.

In [18]:
response = await router.handle(query="günstige Windeln")
pretty(response)

Response type:  recommendations
Language:       de
Intent:         search
Specificity:    specific (confidence 0.95)
Extracted query:  günstige Windeln
Target partner: None
Is basket:      False
Deals preferred: True
Reasoning:      The query is in German, targets a specific product category (diapers) with a clear price preference signal ("günstig"), making it a specific search with deal-seeking intent; no partner is explicitly mentioned.
Latency:        7708 ms
Cost:           €0.00320

Top 5 recommendations:
  1. [edeka ] Premium Rindersteak 300g                                (sem=0.35, loy=1.10, div=0.90, final=0.63)
  2. [dm    ] Pampers Premium Protection Gr. 3                        (sem=0.41, loy=0.70, div=0.20, final=0.48)
  3. [dm    ] babylove Windeln Größe 4                                (sem=0.50, loy=0.30, div=0.20, final=0.41)
  4. [amazon] Adidas Ultraboost 22 Laufschuhe Herren                  (sem=0.34, loy=0.30, div=0.90, final=0.38)
  5. [dm    ] Pampers Premium Pr

In [20]:
response = await router.handle(query="Bitte zeige mir Angebote für günstige Windeln", user_context=edeka_heavy)
pretty(response)

Response type:  recommendations
Language:       de
Intent:         search
Specificity:    specific (confidence 0.95)
Extracted query:  günstige Windeln
Target partner: None
Is basket:      False
Deals preferred: True
Reasoning:      The query is in German, targets a concrete product ("Windeln") with explicit deal-seeking language ("Angebote", "günstig"), making it a specific search with deal preference; no partner is mentioned, so target_partner is null and it is a single-item query.
Latency:        5084 ms
Cost:           €0.00320

Top 5 recommendations:
  1. [edeka ] Premium Rindersteak 300g                                (sem=0.35, loy=1.10, div=0.15, final=0.56)
  2. [dm    ] Pampers Premium Protection Gr. 3                        (sem=0.41, loy=0.70, div=0.40, final=0.50)
  3. [dm    ] babylove Windeln Größe 4                                (sem=0.50, loy=0.30, div=0.40, final=0.43)
  4. [dm    ] Pampers Premium Protection Gr. 3                        (sem=0.45, loy=0.30, div=0.40

## Query 6 — Navigational without partner → polish branch

When the user expresses intent to navigate ("take me to the shop") but does NOT name
a partner, the system should ask which partner. This is a hardcoded fast-path —
no second LLM call, no retrieval — because the answer is fully determined.

Watch for:
- Latency dramatically lower than basket queries (only one LLM call)
- Hardcoded clarifying question + options

In [11]:
response = await router.handle(query="take me to the shop")
pretty(response)

Response type:  clarification
Language:       en
Intent:         discovery
Specificity:    navigational (confidence 0.60)
Extracted query:  
Target partner: None
Is basket:      False
Deals preferred: False
Reasoning:      The user wants to navigate to a shop but doesn't specify which partner or product, making it navigational; no recognized partner is mentioned, no deal-seeking language, and no specific product is implied.
Latency:        3438 ms
Cost:           €0.00320

Clarifying question: Which partner would you like to visit?
Options: ['dm', 'EDEKA', 'Amazon']


## Query 7 — Support intent → out-of-scope

Queries about points, accounts, or service get classified as `intent=support` and
short-circuited to a graceful out-of-scope response pointing at PAYBACK support.
Honest scope boundary rather than fake clarification.

In [12]:
response = await router.handle(query="how do I redeem my points")
pretty(response)

Response type:  clarification
Language:       en
Intent:         support
Specificity:    vague (confidence 0.97)
Extracted query:  redeem points
Target partner: None
Is basket:      False
Deals preferred: False
Reasoning:      The user is asking a how-to question about the PAYBACK points redemption process — a clear account/program support query, not a product search.
Latency:        4244 ms
Cost:           €0.00320

Clarifying question: I can help you find products across dm, EDEKA, and Amazon — but for points, account, or service questions, please visit https://www.payback.group/en/contact or contact PAYBACK support.
Options: ['Find a product instead', 'Browse deals', 'Go to https://www.payback.group/en/contact']


## Quick verdict on what's built

After running the seven queries above, here's an honest summary of what works and
what's limited.

### What works

- **Synthetic catalog generation** with batched LLM tool-use, per-item Pydantic
  validation, and a per-batch acceptance threshold
- **Multilingual retrieval** with a regression test enforcing cross-lingual capability
- **Intent classification** with refined `target_partner` semantics — partner extraction
  on any recognized partner mention, null fallback for unrecognized partners (REWE, Lidl)
- **Catalog-grounded query expansion** with pre-flight category discovery and
  per-sub-query relevance filtering, with dropped-query transparency
- **Loyalty ranker** with three weighted signals (semantic + commercial + diversity)
  and cold-start fallback to result-set diversity
- **Router** with five branches including polished out-of-scope and partner-picker
  fast paths
- **Three-metric evaluation** — intent accuracy, retrieval precision@5/recall@5,
  end-to-end LLM-as-judge with Claude Opus + judge-vs-human agreement validation
- **Honest observability** — every layer reports its decisions, dropped queries,
  costs, and latencies in the response metadata

### What's limited

**Synthetic catalog coverage** : the ~700 synthetic products cover
dm/EDEKA/Amazon for their real-world categories, with limited cross-partner overlap.
Queries for sparse categories surface semantically-adjacent products rather than
literal matches (e.g. `Schokolade` returns chocolate-flavored yogurts and ice creams —
no chocolate bars in the catalog). The system is doing the right thing on what's there;
the limitation is data, not the system.

**Multilingual embedding model behavior** : the current model
(`paraphrase-multilingual-MiniLM-L12-v2`, 384-dim) handles simple, concrete bilingual
terms well (`Windeln`, `Bio Olivenöl`) but struggles on abstract compound German nouns
(`Grundausstattung`, `Heimtextilien`, `Aufbewahrung`). For such queries, pre-flight
retrieval returns weakly-matched items from adjacent categories. The honest fix might be the
larger multilingual model (e.g. `paraphrase-multilingual-mpnet-base-v2` or Vertex AI's
`text-multilingual-embedding-002`), swappable via the `Embedder` interface.

**Single LLM provider for judge eval**: judge uses Claude Opus, system uses Claude
Sonnet. Same family, different size — reduces but doesn't eliminate self-preference
bias. Production would cross-evaluate against a different provider (e.g. GPT-4); the abstract LLMClient interface makes this a single class addition.

## Evaluation results

Three metrics, all reproducible:

In [13]:
import glob
import os

results_dirs = sorted(glob.glob("evals/results/*"))

print("Evaluation reports available:")
for d in results_dirs[-3:]:
    name = os.path.basename(d)
    print(f"  evals/results/{name}/report.md")

print("\nFor full results, open each report.md file.")
print("\nKey metrics to look for:")
print("  Intent:     overall accuracy, per-dimension breakdown, failure clusters")
print("  Retrieval:  mean precision@5, mean recall@5, weakest query class")
print("  E2E:        mean total score (X/9), per-dimension means, judge-human agreement")

Evaluation reports available:
  evals/results/e2e_20260514_162653/report.md
  evals/results/intent_20260514_144702/report.md
  evals/results/retrieval_20260514_145426/report.md

For full results, open each report.md file.

Key metrics to look for:
  Intent:     overall accuracy, per-dimension breakdown, failure clusters
  Retrieval:  mean precision@5, mean recall@5, weakest query class
  E2E:        mean total score (X/9), per-dimension means, judge-human agreement


## What would change in production

If this system were going to PAYBACK production, the priorities would be:

1. **Real partner catalogs** — replace synthetic data; I expect retrieval quality to
   improve substantially when product descriptions reflect real merchandising
2. **Better multilingual embedding** — I would swap to a stronger model (768-dim+)  for German abstract-noun handling
3. **Vertex AI Vector Search** — at PAYBACK scale (millions of products + users),
   colocate embeddings with the existing BigQuery footprint (see `bigquery_store.py`
   for the production stub) --->> ADR-016
4. **Real user profiles** — replacing mock JSON profiles with a call to PAYBACK's
   user-profile service; the `UserContext` schema and ranker code stay unchanged
5. **Cross-provider judge eval** — adding a second judge (e.g. GPT-4 but for this demo used Opus) for
   bias-controlled evaluation; the abstract `LLMClient` interface makes this
   a single class addition

## Repository

- `app/` — production code
- `evals/` — three evaluation runners + labeled datasets
- `data/` — synthetic catalogs (~700 products) + mock user profiles
- `tests/` — pytest suite covering router, ranker, retrieval, intent, expansion
- `docs/decisions.md` — Architecture Decision Records (ADR-001 through ADR-010)
- `scripts/try_router.py`, `scripts/try_api.py` — manual smoke tests
- `README.md` — quick start, demo queries, architecture diagram